In [2]:
#### Libraries ###
import os
import glob
import pandas as pd
import numpy as np
import scanpy as sc

import scipy.sparse as sp  
from scipy.sparse import csr_matrix
from scipy.io import mmread

import matplotlib.pyplot as plt

os.chdir("/home/ascott10/documents/projects/rnaseq-pipe/")

In [26]:
from pybiomart import Dataset

dataset = Dataset(name='hsapiens_gene_ensembl', host='http://www.ensembl.org')



In [28]:

mapping = dataset.query(
    attributes=['ensembl_gene_id', 'hgnc_symbol']
)

In [29]:


mapping_dict = dict(
    zip(mapping['Gene stable ID'], mapping['HGNC symbol'])
)

In [41]:
df = pd.DataFrame(mapping_dict.items(), columns=["ensembl_id", "hgnc_symbol"])
df.to_csv("reference_data_common/ensembl_to_hgnc")

In [40]:
df

,ensembl_id,hgnc_symbol
0,ENSG00000210049,MT-TF
1,ENSG00000211459,MT-RNR1
2,ENSG00000210077,MT-TV
3,ENSG00000210082,MT-RNR2
4,ENSG00000209082,MT-TL1
...,...,...
86364,ENSG00000168710,AHCYL1
86365,ENSG00000081692,JMJD4
86366,ENSG00000157873,TNFRSF14
86367,ENSG00000132676,DAP3


In [ ]:
mtx_file = "mtab_tumors/raw_data/E-MTAB-8559.aggregated_filtered_counts.mtx"
cells_file = "mtab_tumors/raw_data/E-MTAB-8559.aggregated_filtered_counts.mtx_cols"
genes_file = "mtab_tumors/raw_data/E-MTAB-8559.aggregated_filtered_counts.mtx_rows"
# Load matrix
adata = sc.read_mtx(mtx_file).T

# Load genes (23,284 rows)
genes = pd.read_csv(
    genes_file,
    sep="\t",
    header=None
)

# Load cells (19,880 rows)
cells = pd.read_csv(cells_file, header=None)

# Assign correctly:
adata.var_names = genes[0].astype(str)   # genes to var
adata.obs_names = cells[0].astype(str)   # cells to obs

print("Genes assigned:", adata.var_names[:5])
print("Cells assigned:", adata.obs_names[:5])

gene_map = pd.read_csv("reference_data_common/ensembl_to_hgnc")  # must have: ensembl_gene_id, hgnc_symbol

if "hgnc_symbol" in gene_map:

    ensg_to_hgnc = dict(zip(gene_map["ensembl_id"], gene_map["hgnc_symbol"]))
    adata.var["ensembl_id"] = adata.var_names.astype(str)
    adata.var["gene_symbol"] = adata.var["ensembl_id"].map(ensg_to_hgnc)
    adata.var["gene_symbol"] = adata.var["gene_symbol"].fillna(adata.var["ensembl_id"])
    adata.var_names = adata.var["gene_symbol"].astype(str)

adata.var_names_make_unique()
adata.var.index.name = None
print("Genes names: ", adata.var_names[:10])


Genes assigned: Index(['ENSG00000000003', 'ENSG00000000419', 'ENSG00000000457',
       'ENSG00000000460', 'ENSG00000000938'],
      dtype='object')
Cells assigned: Index(['SAMEA6492740-AAACCCACAGTTAGGG', 'SAMEA6492740-AAACCCACATGTGTCA',
       'SAMEA6492740-AAACCCAGTCGCATGC', 'SAMEA6492740-AAACCCAGTCTTTCAT',
       'SAMEA6492740-AAACCCATCCGTGTCT'],
      dtype='object')
Genes names:  Index(['TSPAN6', 'DPM1', 'SCYL3', 'FIRRM', 'FGR', 'CFH', 'FUCA2', 'GCLC',
       'NFYA', 'STPG1'],
      dtype='object')


KeyError: 'sample'

In [62]:
meta_df = pd.read_csv("mtab_tumors/reference_data_mtab_tumors/sample_metadata.csv")
meta_df

,donor,sample_id,sex,age,FIGO_grade
0,SAMEA6492740,38b,F,81,stage 3c
1,SAMEA6492741,59,F,46,stage 3c
2,SAMEA6492742,74-1,F,73,stage 3b
3,SAMEA6492743,79,F,73,stage 3c


In [63]:

adata.obs["barcode"] = adata.obs.index
adata.obs["donor"] = adata.obs["barcode"].str.split("-").str[0]

adata.obs


,barcode,donor
SAMEA6492740-AAACCCACAGTTAGGG,SAMEA6492740-AAACCCACAGTTAGGG,SAMEA6492740
SAMEA6492740-AAACCCACATGTGTCA,SAMEA6492740-AAACCCACATGTGTCA,SAMEA6492740
SAMEA6492740-AAACCCAGTCGCATGC,SAMEA6492740-AAACCCAGTCGCATGC,SAMEA6492740
SAMEA6492740-AAACCCAGTCTTTCAT,SAMEA6492740-AAACCCAGTCTTTCAT,SAMEA6492740
SAMEA6492740-AAACCCATCCGTGTCT,SAMEA6492740-AAACCCATCCGTGTCT,SAMEA6492740
...,...,...
SAMEA6492743-TTTGGTTTCATCGTAG,SAMEA6492743-TTTGGTTTCATCGTAG,SAMEA6492743
SAMEA6492743-TTTGTTGCAACCGACC,SAMEA6492743-TTTGTTGCAACCGACC,SAMEA6492743
SAMEA6492743-TTTGTTGGTCAGTCCG,SAMEA6492743-TTTGTTGGTCAGTCCG,SAMEA6492743
SAMEA6492743-TTTGTTGGTCCTGGTG,SAMEA6492743-TTTGTTGGTCCTGGTG,SAMEA6492743


In [ ]:
adata.obs = adata.obs.join(meta_df, on = "donor")

In [64]:
adata.obs = adata.obs.merge(meta_df, on = "donor", how = "left")

adata.obs

,barcode,donor,sample_id,sex,age,FIGO_grade
0,SAMEA6492740-AAACCCACAGTTAGGG,SAMEA6492740,38b,F,81,stage 3c
1,SAMEA6492740-AAACCCACATGTGTCA,SAMEA6492740,38b,F,81,stage 3c
2,SAMEA6492740-AAACCCAGTCGCATGC,SAMEA6492740,38b,F,81,stage 3c
3,SAMEA6492740-AAACCCAGTCTTTCAT,SAMEA6492740,38b,F,81,stage 3c
4,SAMEA6492740-AAACCCATCCGTGTCT,SAMEA6492740,38b,F,81,stage 3c
...,...,...,...,...,...,...
19875,SAMEA6492743-TTTGGTTTCATCGTAG,SAMEA6492743,79,F,73,stage 3c
19876,SAMEA6492743-TTTGTTGCAACCGACC,SAMEA6492743,79,F,73,stage 3c
19877,SAMEA6492743-TTTGTTGGTCAGTCCG,SAMEA6492743,79,F,73,stage 3c
19878,SAMEA6492743-TTTGTTGGTCCTGGTG,SAMEA6492743,79,F,73,stage 3c
